# Análise Exploratória das Variáveis SIM para o Modelo Dimensional

**Objetivo:** Identificar os valores distintos de cada variável do SIM que será usada nas dimensões do star schema, verificando:
- Valores únicos (categorias reais)
- Valores ausentes (nulos/vazios)
- Compatibilidade entre anos (2014-2023)
- Distribuição de frequência

**Variáveis-alvo por dimensão:**
| Dimensão | Coluna(s) SIM |
|---|---|
| Dim_Tempo | DTOBITO |
| Dim_Municipio | CODMUNOCOR, CODMUNRES |
| Dim_FaixaEtaria | IDADE |
| Dim_RacaCor | RACACOR |
| Dim_LocalOcorrencia | LOCOCOR |
| Dim_SituacaoGestacionalObito | TPMORTEOCO |
| Dim_TipoParto | PARTO |
| Dim_MomentoObitoParto | OBITOPARTO |
| Dim_TipoGravidez | GRAVIDEZ |
| Dim_SemanaGestacao | SEMAGESTAC |
| Dim_CID | CAUSABAS, CAUSAMAT, CAUSABAS_O |
| Dim_EstabelecimentoSaude | CODESTAB |
| Atributo direto na fato | SEXO, ASSISTMED |

In [1]:
import pandas as pd
import os
import glob

print("pandas version:", pd.__version__)

pandas version: 3.0.3


In [2]:
# Configuração: pasta dos dados SIM
pasta_sim = "../arquivos/SIM"

# Lista todos os CSVs anuais, ordenados
arquivos = sorted(glob.glob(os.path.join(pasta_sim, "dados_*.csv")))
print(f"Arquivos encontrados: {len(arquivos)}")
for a in arquivos:
    print(f"  {os.path.basename(a)}")

Arquivos encontrados: 13
  dados_2014.csv
  dados_2015.csv
  dados_2016.csv
  dados_2017.csv
  dados_2018.csv
  dados_2019.csv
  dados_2020.csv
  dados_2021.csv
  dados_2022.csv
  dados_2023.csv
  dados_2024.csv
  dados_2025.csv
  dados_2026.csv


In [3]:
# Colunas de interesse para o modelo dimensional
COLUNAS_INTERESSE = [
    'DTOBITO',       # Dim_Tempo
    'SEXO',          # Atributo direto na fato (normalizado)
    'IDADE',         # Dim_FaixaEtaria
    'RACACOR',       # Dim_RacaCor
    'LOCOCOR',       # Dim_LocalOcorrencia
    'TPMORTEOCO',    # Dim_SituacaoGestacionalObito
    'PARTO',         # Dim_TipoParto
    'OBITOPARTO',    # Dim_MomentoObitoParto
    'GRAVIDEZ',      # Dim_TipoGravidez
    'SEMAGESTAC',    # Dim_SemanaGestacao
    'CAUSABAS',      # Dim_CID (causa básica)
    'CAUSAMAT',      # Dim_CID (causa materna)
    'CAUSABAS_O',    # Dim_CID (causa básica original)
    'CODMUNOCOR',    # Dim_Municipio (ocorrência)
    'CODMUNRES',     # Dim_Municipio (residência)
    'CODESTAB',      # Dim_EstabelecimentoSaude
    'ASSISTMED',     # recebeu_assistencia_medica
]

In [4]:
# Função para ler e consolidar todos os anos com apenas as colunas de interesse
def carregar_dados_sim(arquivos, colunas):
    """Lê CSVs do SIM, normaliza colunas para maiúsculo e retorna DataFrame consolidado."""
    frames = []
    for arquivo in arquivos:
        ano = os.path.basename(arquivo).replace('dados_', '').replace('.csv', '')
        print(f"Lendo {os.path.basename(arquivo)}...", end=' ')
        df_ano = pd.read_csv(arquivo, sep=';', encoding='latin1', dtype=str, low_memory=False)
        df_ano.columns = df_ano.columns.str.upper()
        
        # Verifica quais colunas de interesse existem neste arquivo
        cols_presentes = [c for c in colunas if c in df_ano.columns]
        cols_ausentes = [c for c in colunas if c not in df_ano.columns]
        
        if cols_ausentes:
            print(f"(faltam: {cols_ausentes})", end=' ')
        
        df_ano = df_ano[cols_presentes].copy()
        df_ano['_ano'] = ano
        frames.append(df_ano)
        print(f"{len(df_ano)} registros")
    
    df = pd.concat(frames, ignore_index=True)
    print(f"\nTotal consolidado: {len(df)} registros ({len(frames)} anos)")
    return df

df_sim = carregar_dados_sim(arquivos, COLUNAS_INTERESSE)

Lendo dados_2014.csv... 1227039 registros
Lendo dados_2015.csv... 1264175 registros
Lendo dados_2016.csv... 1309774 registros
Lendo dados_2017.csv... 1312663 registros
Lendo dados_2018.csv... 1316719 registros
Lendo dados_2019.csv... 1349801 registros
Lendo dados_2020.csv... 1556824 registros
Lendo dados_2021.csv... 1832649 registros
Lendo dados_2022.csv... 1544266 registros
Lendo dados_2023.csv... 1465610 registros
Lendo dados_2024.csv... 1532015 registros
Lendo dados_2025.csv... 1528085 registros
Lendo dados_2026.csv... 506531 registros

Total consolidado: 17746151 registros (13 anos)


## 1. SEXO — Valores Originais no SIM

Mapeamento definido no plano:
| Original (SIM) | Normalizado |
|---|---|
| 'F', '2' | 'F' |
| 'M', '1' | 'M' |
| 'I', '0', '9', outros | 'I' |

Vamos verificar os valores reais presentes.

In [5]:
# --- SEXO ---
print("=" * 60)
print("SEXO - Valores originais no SIM")
print("=" * 60)

sexo_counts = df_sim['SEXO'].value_counts(dropna=False).sort_index()
print(sexo_counts.to_string())
print(f"\nTotal: {sexo_counts.sum()}")
print(f"Nulos: {df_sim['SEXO'].isna().sum()}")

# Aplicar a normalização proposta
def normaliza_sexo(valor):
    if pd.isna(valor):
        return 'I'
    valor = str(valor).strip().upper()
    if valor in ('F', '2'):
        return 'F'
    elif valor in ('M', '1'):
        return 'M'
    else:
        return 'I'

df_sim['sexo_normalizado'] = df_sim['SEXO'].apply(normaliza_sexo)
print("\n--- Distribuição após normalização ---")
print(df_sim['sexo_normalizado'].value_counts().to_string())

SEXO - Valores originais no SIM
SEXO
0       7464
1    9835350
2    7903337

Total: 17746151
Nulos: 0

--- Distribuição após normalização ---
sexo_normalizado
M    9835350
F    7903337
I       7464


In [6]:
# Verificar consistência do SEXO por ano
print("SEXO por ano (valores originais):\n")
print(pd.crosstab(df_sim['_ano'], df_sim['SEXO'], margins=True).to_string())

SEXO por ano (valores originais):

SEXO     0        1        2       All
_ano                                  
2014   755   693922   532362   1227039
2015   675   709117   554383   1264175
2016   573   736842   572359   1309774
2017   621   734469   577573   1312663
2018   646   733616   582457   1316719
2019   557   745519   603725   1349801
2020   630   874167   682027   1556824
2021   683  1015350   816616   1832649
2022   626   844920   698720   1544266
2023   526   803200   661884   1465610
2024   511   836659   694845   1532015
2025   501   832582   695002   1528085
2026   160   274987   231384    506531
All   7464  9835350  7903337  17746151


## 2. IDADE — Faixa Etária

O campo IDADE no SIM tem 3 dígitos:
- 1º dígito: unidade de medida (`4` = anos)
- 2º e 3º dígitos: valor

Faixas previstas: 10-14, 15-19, 20-24, 25-29, 30-34, 35-39, 40-44, 45-49

O filtro de mortalidade materna usa `IDADE BETWEEN '410' AND '449'` (10 a 49 anos).

In [7]:
# --- IDADE ---
print("=" * 60)
print("IDADE - Valores originais no SIM")
print("=" * 60)

# Mostrar distribuição completa
idade_counts = df_sim['IDADE'].value_counts(dropna=False).sort_index()
print(f"Total de valores distintos: {len(idade_counts)}")
print(f"\nPrimeiros 30 valores:")
print(idade_counts.head(30).to_string())
print(f"\nÚltimos 30 valores:")
print(idade_counts.tail(30).to_string())

# Verificar os primeiros dígitos (unidade de medida)
print("\n\n--- Primeiro dígito do IDADE (unidade de medida) ---")
primeiro_digito = df_sim['IDADE'].dropna().str[0].value_counts().sort_index()
print(primeiro_digito.to_string())

IDADE - Valores originais no SIM
Total de valores distintos: 257

Primeiros 30 valores:
IDADE
001    2628
002     786
003     582
004     314
005    2137
006     260
007     253
008     287
009     216
010    2299
011     226
012     300
013     270
014     243
015    1482
016     243
017     290
018     286
019     223
020    2106
021     304
022     373
023     395
024     300
025     961
026     307
027     333
028     339
029     261
030    2733

Últimos 30 valores:
IDADE
502    20761
503    15006
504    10688
505     7378
506     5138
507     3813
508     2753
509     1913
510     1311
511      925
512      644
513      479
514      315
515      212
516      144
517       92
518       54
519       31
520       20
521       15
522       13
523       10
524        7
525        3
526        5
527        1
529        1
531        1
533        1
999    30856


--- Primeiro dígito do IDADE (unidade de medida) ---
IDADE
0       33113
1       64978
2      194876
3      124601
4    1715873

In [8]:
# IDADE: analisar apenas registros com 1º dígito = 4 (medida em anos)
# e verificar a distribuição de idade (2 últimos dígitos)
df_idade_anos = df_sim[df_sim['IDADE'].str[0] == '4'].copy()
df_idade_anos['idade_valor'] = df_idade_anos['IDADE'].str[1:3].astype(int)

print("Distribuição de idade (anos) - apenas registros com unidade=4:\n")
print(df_idade_anos['idade_valor'].value_counts().sort_index().to_string())

# Faixas etárias conforme plano
faixas = [
    (10, 14, '10-14'),
    (15, 19, '15-19'),
    (20, 24, '20-24'),
    (25, 29, '25-29'),
    (30, 34, '30-34'),
    (35, 39, '35-39'),
    (40, 44, '40-44'),
    (45, 49, '45-49'),
    (50, 200, '50+'),
]

def classificar_faixa(idade):
    if pd.isna(idade):
        return None
    for min_id, max_id, nome in faixas:
        if min_id <= idade <= max_id:
            return nome
    return 'DESCONHECIDA'

df_idade_anos['faixa_etaria'] = df_idade_anos['idade_valor'].apply(classificar_faixa)
print("\n--- Distribuição por faixa etária ---")
print(df_idade_anos['faixa_etaria'].value_counts().to_string())

Distribuição de idade (anos) - apenas registros com unidade=4:

idade_valor
0         35
1      30356
2      16882
3      12181
4      10359
5       8829
6       8098
7       7562
8       7200
9       7480
10      7649
11      7976
12      9270
13     11659
14     16537
15     23927
16     33959
17     44231
18     52684
19     58455
20     62752
21     63704
22     63783
23     63994
24     63752
25     64401
26     64586
27     65138
28     66103
29     66901
30     68572
31     69788
32     72089
33     74874
34     77190
35     81705
36     84514
37     88340
38     91820
39     95008
40    100322
41    104490
42    109281
43    113286
44    118735
45    124036
46    129825
47    136268
48    143455
49    150344
50    160890
51    169145
52    177036
53    187205
54    197329
55    208406
56    220397
57    233643
58    243972
59    255257
60    267092
61    277540
62    287849
63    298477
64    307563
65    318654
66    329478
67    337319
68    343799
69    349387
70    356425
7

## 3. RACACOR — Raça/Cor

| Código | Descrição |
|---|---|
| 1 | Branca |
| 2 | Preta |
| 3 | Amarela |
| 4 | Parda |
| 5 | Indígena |
| 9 | Ignorado |

In [9]:
# --- RACACOR ---
print("=" * 60)
print("RACACOR - Raça/Cor")
print("=" * 60)

racacor_counts = df_sim['RACACOR'].value_counts(dropna=False).sort_index()
print(racacor_counts.to_string())
print(f"\nNulos: {df_sim['RACACOR'].isna().sum()}")
print(f"Valores distintos: {racacor_counts.index.tolist()}")

RACACOR - Raça/Cor
RACACOR
1      8981870
2      1459994
3       105099
4      6677636
5        58519
9            5
NaN     463028

Nulos: 463028
Valores distintos: ['1', '2', '3', '4', '5', '9', nan]


## 4. LOCOCOR — Local de Ocorrência

| Código | Descrição |
|---|---|
| 1 | Hospital |
| 2 | Outros est. saúde |
| 3 | Domicílio |
| 4 | Via pública |
| 5 | Outros |
| 6 | Aldeia indígena |
| 9 | Ignorado |

In [10]:
# --- LOCOCOR ---
print("=" * 60)
print("LOCOCOR - Local de Ocorrência")
print("=" * 60)

lococor_counts = df_sim['LOCOCOR'].value_counts(dropna=False).sort_index()
print(lococor_counts.to_string())
print(f"\nNulos: {df_sim['LOCOCOR'].isna().sum()}")
print(f"Valores distintos: {sorted(lococor_counts.index[~pd.isna(lococor_counts.index)].astype(str).tolist())}")

LOCOCOR - Local de Ocorrência
LOCOCOR
1      11844727
2       1124374
3       3518088
4        670141
5        574702
6          1095
9         13021
NaN           3

Nulos: 3
Valores distintos: ['1', '2', '3', '4', '5', '6', '9']


## 5. TPMORTEOCO — Situação Gestacional do Óbito

| Código | Descrição |
|---|---|
| 1 | Na gravidez |
| 2 | No parto |
| 3 | No abortamento |
| 4 | Até 42 dias pós-parto |
| 5 | 43d a 1 ano pós-parto |
| 8 | Não ocorreu nestes períodos |
| 9 | Ignorado (inclui 6 e 7 normalizados) |

> Nota: Valores 6 e 7 encontrados nos dados serão normalizados para 9 (Ignorado) no ETL

In [11]:
# --- TPMORTEOCO ---
print("=" * 60)
print("TPMORTEOCO - Situação Gestacional do Óbito")
print("=" * 60)

tpmorteoco_counts = df_sim['TPMORTEOCO'].value_counts(dropna=False).sort_index()
print(tpmorteoco_counts.to_string())
print(f"\nNulos: {df_sim['TPMORTEOCO'].isna().sum()}")
print(f"Valores distintos: {sorted(tpmorteoco_counts.index[~pd.isna(tpmorteoco_counts.index)].astype(str).tolist())}")

TPMORTEOCO - Situação Gestacional do Óbito
TPMORTEOCO
1          7879
2          2243
3           906
4         14162
5          8075
6            29
7             4
8        898470
9        180631
NaN    16633752

Nulos: 16633752
Valores distintos: ['1', '2', '3', '4', '5', '6', '7', '8', '9']


## 6. PARTO — Tipo de Parto

| Código | Descrição |
|---|---|
| 1 | Vaginal |
| 2 | Cesáreo |
| 9 | Ignorado |

In [12]:
# --- PARTO ---
print("=" * 60)
print("PARTO - Tipo de Parto")
print("=" * 60)

parto_counts = df_sim['PARTO'].value_counts(dropna=False).sort_index()
print(parto_counts.to_string())
print(f"\nNulos: {df_sim['PARTO'].isna().sum()}")
print(f"Valores distintos: {sorted(parto_counts.index[~pd.isna(parto_counts.index)].astype(str).tolist())}")

PARTO - Tipo de Parto
PARTO
1        185400
2        193785
9          5752
NaN    17361214

Nulos: 17361214
Valores distintos: ['1', '2', '9']


## 7. OBITOPARTO — Momento do Óbito em relação ao Parto

| Código | Descrição |
|---|---|
| 1 | Antes |
| 2 | Durante |
| 3 | Depois |
| 9 | Ignorado |

In [13]:
# --- OBITOPARTO ---
print("=" * 60)
print("OBITOPARTO - Momento do Óbito em relação ao Parto")
print("=" * 60)

obitoparto_counts = df_sim['OBITOPARTO'].value_counts(dropna=False).sort_index()
print(obitoparto_counts.to_string())
print(f"\nNulos: {df_sim['OBITOPARTO'].isna().sum()}")
print(f"Valores distintos: {sorted(obitoparto_counts.index[~pd.isna(obitoparto_counts.index)].astype(str).tolist())}")

OBITOPARTO - Momento do Óbito em relação ao Parto
OBITOPARTO
1             6
2             7
3        373255
9          8429
NaN    17364454

Nulos: 17364454
Valores distintos: ['1', '2', '3', '9']


## 8. GRAVIDEZ — Tipo de Gravidez

| Código | Descrição |
|---|---|
| 1 | Única |
| 2 | Dupla |
| 3 | Tripla e mais |
| 9 | Ignorada |

In [14]:
# --- GRAVIDEZ ---
print("=" * 60)
print("GRAVIDEZ - Tipo de Gravidez")
print("=" * 60)

gravidez_counts = df_sim['GRAVIDEZ'].value_counts(dropna=False).sort_index()
print(gravidez_counts.to_string())
print(f"\nNulos: {df_sim['GRAVIDEZ'].isna().sum()}")
print(f"Valores distintos: {sorted(gravidez_counts.index[~pd.isna(gravidez_counts.index)].astype(str).tolist())}")

GRAVIDEZ - Tipo de Gravidez
GRAVIDEZ
1        345303
2         34939
3          2222
9          4444
NaN    17359243

Nulos: 17359243
Valores distintos: ['1', '2', '3', '9']


## 9. SEMAGESTAC — Semana de Gestação

| Faixa | Semanas |
|---|---|
| 0-21 | 0 a 21 |
| 22-27 | 22 a 27 |
| 28-31 | 28 a 31 |
| 32-36 | 32 a 36 |
| 37-41 | 37 a 41 |
| 42+ | 42 ou mais |

In [15]:
# --- SEMAGESTAC ---
print("=" * 60)
print("SEMAGESTAC - Semana de Gestação")
print("=" * 60)

semagestac_counts = df_sim['SEMAGESTAC'].value_counts(dropna=False).sort_index()
print(semagestac_counts.to_string())
print(f"\nNulos: {df_sim['SEMAGESTAC'].isna().sum()}")
print(f"Valores distintos: {sorted(semagestac_counts.index[~pd.isna(semagestac_counts.index)].astype(str).tolist())}")

SEMAGESTAC - Semana de Gestação
SEMAGESTAC
0          2589
1          2849
10           83
11           49
12           93
13           21
14           44
15           56
16          166
17          199
18          434
19          949
2          1925
20         3547
21         5623
22        10836
23        14354
24        18467
25        17462
26        19272
27        17089
28        17587
29        13368
3          1300
30        13307
31        11694
32        13366
33        12022
34        13405
35        13611
36        17211
37        21829
38        27483
39        30377
4           685
40        21658
41         7689
42         1955
43          147
44           60
45            1
5           367
6           239
7           163
8           138
9           319
99         7213
NaN    17382850

Nulos: 17382850
Valores distintos: ['0', '1', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '2', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '3', '30', '31', 

## 10. CID — Causas (Role-Playing Dimension)

Três colunas de CID que apontam para a mesma Dim_CID:
- **CAUSABAS**: Causa básica (NOT NULL na fato)
- **CAUSAMAT**: Causa materna
- **CAUSABAS_O**: Causa básica original

In [16]:
# --- CAUSABAS (causa básica) ---
print("=" * 60)
print("CAUSABAS - Causa Básica (NOT NULL)")
print("=" * 60)

causabas_counts = df_sim['CAUSABAS'].value_counts(dropna=False).sort_index()
print(f"Total de CIDs distintos: {len(causabas_counts)}")
print(f"Nulos: {df_sim['CAUSABAS'].isna().sum()}")
print(f"\nTop 20 CIDs mais frequentes:")
print(causabas_counts.head(20).to_string())
print(f"\nBottom 10 CIDs menos frequentes:")
print(causabas_counts.tail(10).to_string())

CAUSABAS - Causa Básica (NOT NULL)
Total de CIDs distintos: 8067
Nulos: 0

Top 20 CIDs mais frequentes:
CAUSABAS
A010       7
A011       1
A014       3
A020      64
A021     235
A022      18
A028       4
A029      25
A031       1
A033       1
A038       1
A039      14
A040      59
A041      18
A042      19
A043       9
A044      69
A045       4
A046       6
A047    2855

Bottom 10 CIDs menos frequentes:
CAUSABAS
Y870      92
Y871     568
Y872     650
Y880      44
Y881      36
Y882      33
Y883     656
Y890      17
Y891       2
Y899    1231


In [17]:
# --- CAUSAMAT (causa materna) ---
print("=" * 60)
print("CAUSAMAT - Causa Materna")
print("=" * 60)

causamat_counts = df_sim['CAUSAMAT'].value_counts(dropna=False).sort_index()
print(f"Total de CIDs distintos: {len(causamat_counts)}")
print(f"Nulos: {df_sim['CAUSAMAT'].isna().sum()}")
# Muitos nulos são esperados, pois nem todo óbito tem causa materna
nao_nulos = df_sim['CAUSAMAT'].notna().sum()
print(f"Não nulos: {nao_nulos} ({nao_nulos/len(df_sim)*100:.1f}%)")
print(f"\nTop 20 CIDs maternos mais frequentes:")
print(causamat_counts.head(20).to_string())

CAUSAMAT - Causa Materna
Total de CIDs distintos: 11
Nulos: 17745816
Não nulos: 335 (0.0%)

Top 20 CIDs maternos mais frequentes:
CAUSAMAT
O930         102
O931           7
O932           1
O933           4
O934          18
O935           6
O936          58
O937         119
O938           9
O939          11
NaN     17745816


In [18]:
# --- CAUSABAS_O (causa básica original) ---
print("=" * 60)
print("CAUSABAS_O - Causa Básica Original")
print("=" * 60)

causabas_o_counts = df_sim['CAUSABAS_O'].value_counts(dropna=False).sort_index()
print(f"Total de CIDs distintos: {len(causabas_o_counts)}")
print(f"Nulos: {df_sim['CAUSABAS_O'].isna().sum()}")
nao_nulos = df_sim['CAUSABAS_O'].notna().sum()
print(f"Não nulos: {nao_nulos} ({nao_nulos/len(df_sim)*100:.1f}%)")
print(f"\nTop 20 CIDs originais mais frequentes:")
print(causabas_o_counts.head(20).to_string())

CAUSABAS_O - Causa Básica Original
Total de CIDs distintos: 9481
Nulos: 16918
Não nulos: 17729233 (99.9%)

Top 20 CIDs originais mais frequentes:
CAUSABAS_O
6309      1
A00       2
A001      2
A009     11
A010      8
A011      1
A014      2
A02       1
A020     55
A021    225
A022     17
A028      5
A029     22
A03       1
A031      1
A038      1
A039     12
A04       9
A040     54
A041     15


## 11. DTOBITO — Data do Óbito (Dim_Tempo)

Verificar amplitude de datas, anos cobertos e valores malformados.

In [19]:
# --- DTOBITO ---
print("=" * 60)
print("DTOBITO - Data do Óbito")
print("=" * 60)

# DTOBITO está no formato DDMMYYYY (ex: 27092014 = 27/09/2014)
# Primeiro, verificar formatos das datas
print("Amostra de 20 valores de DTOBITO:")
print(df_sim['DTOBITO'].dropna().sample(20, random_state=42).to_string())

# Extrair ano (posições 4-7 do formato DDMMYYYY)
df_sim['ano_dtobito'] = df_sim['DTOBITO'].str[4:8]
print(f"\n--- Distribuição de anos ---")
ano_counts = df_sim['ano_dtobito'].value_counts(dropna=False).sort_index()
print(ano_counts.to_string())

# Verificar valores com formato inesperado
print(f"\n--- Valores com comprimento diferente de 8 ---")
tamanhos = df_sim['DTOBITO'].dropna().str.len().value_counts().sort_index()
print(f"Distribuição de comprimentos: {tamanhos.to_dict()}")

# Verificar nulos
print(f"\nNulos em DTOBITO: {df_sim['DTOBITO'].isna().sum()}")

# Extrair data completa no formato ISO (YYYY-MM-DD)
df_sim['data_iso'] = df_sim['DTOBITO'].apply(
    lambda x: f"{x[4:8]}-{x[2:4]}-{x[0:2]}" if pd.notna(x) and len(str(x)) == 8 else None
)
print(f"\n--- Amostra de datas convertidas para ISO ---")
print(df_sim['data_iso'].dropna().sample(10, random_state=42).to_string())

DTOBITO - Data do Óbito
Amostra de 20 valores de DTOBITO:
6622604     07072019
1713852     09032015
10955431    01062021
282833      06012014
12347152    13082022
7087522     07052019
1494244     25022015
11910405    12022022
933165      03082014
13498554    15072023
12314507    26122022
4903515     20092017
211262      04012014
10934094    04012021
2036374     20062015
6844749     20032019
8963927     24012020
13538985    24072023
13900170    23102023
171219      21112014

--- Distribuição de anos ---
ano_dtobito
2014    1227039
2015    1264175
2016    1309774
2017    1312663
2018    1316719
2019    1349801
2020    1556824
2021    1832649
2022    1544266
2023    1465610
2024    1532015
2025    1528085
2026     506531

--- Valores com comprimento diferente de 8 ---
Distribuição de comprimentos: {8: 17746151}

Nulos em DTOBITO: 0

--- Amostra de datas convertidas para ISO ---
6622604     2019-07-07
1713852     2015-03-09
10955431    2021-06-01
282833      2014-01-06
12347152    2022-08-

## 12. CODMUNOCOR e CODMUNRES — Municípios

A mesma Dim_Municipio é usada como role-playing (ocorrência e residência).

In [20]:
# --- CODMUNOCOR (Município de Ocorrência) ---
print("=" * 60)
print("CODMUNOCOR - Município de Ocorrência")
print("=" * 60)

mun_ocor_counts = df_sim['CODMUNOCOR'].value_counts(dropna=False).sort_index()
print(f"Total de municípios distintos: {len(mun_ocor_counts)}")
print(f"Nulos: {df_sim['CODMUNOCOR'].isna().sum()}")
print(f"\nTop 10 municípios:")
print(mun_ocor_counts.head(10).to_string())
print(f"\nÚltimos 5:")
print(mun_ocor_counts.tail(5).to_string())

# Verificar comprimento do código
print(f"\n--- Comprimento do código ---")
print(df_sim['CODMUNOCOR'].dropna().str.len().value_counts().sort_index().to_string())

CODMUNOCOR - Município de Ocorrência
Total de municípios distintos: 5592
Nulos: 0

Top 10 municípios:
CODMUNOCOR
110000        7
110001     1023
110002     7554
110003      219
110004    15122
110005      807
110006      702
110007      196
110008      421
110009     1297

Últimos 5:
CODMUNOCOR
522200       796
522205       384
522220       236
522230       186
530010    199153

--- Comprimento do código ---
CODMUNOCOR
6    17746151


In [21]:
# --- CODMUNRES (Município de Residência) ---
print("=" * 60)
print("CODMUNRES - Município de Residência")
print("=" * 60)

mun_res_counts = df_sim['CODMUNRES'].value_counts(dropna=False).sort_index()
print(f"Total de municípios distintos: {len(mun_res_counts)}")
print(f"Nulos: {df_sim['CODMUNRES'].isna().sum()}")
print(f"\nTop 10 municípios:")
print(mun_res_counts.head(10).to_string())

# Comparar cobertura
print(f"\n--- Comparação Ocorrência vs Residência ---")
print(f"Municípios apenas em CODMUNOCOR: {len(set(mun_ocor_counts.index) - set(mun_res_counts.index))}")
print(f"Municípios apenas em CODMUNRES: {len(set(mun_res_counts.index) - set(mun_ocor_counts.index))}")
print(f"Municípios em ambos: {len(set(mun_ocor_counts.index) & set(mun_res_counts.index))}")

CODMUNRES - Município de Residência
Total de municípios distintos: 5596
Nulos: 0

Top 10 municípios:
CODMUNRES
110000     199
110001    1816
110002    7308
110003     429
110004    6736
110005    1377
110006    1317
110007     467
110008     838
110009    2241

--- Comparação Ocorrência vs Residência ---
Municípios apenas em CODMUNOCOR: 0
Municípios apenas em CODMUNRES: 4
Municípios em ambos: 5592


## 13. CODESTAB — Estabelecimento de Saúde (CNES)

Código do estabelecimento para vincular com Dim_EstabelecimentoSaude.

In [22]:
# --- CODESTAB ---
print("=" * 60)
print("CODESTAB - Estabelecimento de Saúde")
print("=" * 60)

codestab_counts = df_sim['CODESTAB'].value_counts(dropna=False).sort_index()
print(f"Total de estabelecimentos distintos: {len(codestab_counts)}")
print(f"Nulos: {df_sim['CODESTAB'].isna().sum()}")
nao_nulos = df_sim['CODESTAB'].notna().sum()
print(f"Não nulos: {nao_nulos} ({nao_nulos/len(df_sim)*100:.1f}%)")
print(f"\nTop 10 estabelecimentos:")
print(codestab_counts.head(10).to_string())
print(f"\nComprimento do código:")
print(df_sim['CODESTAB'].dropna().str.len().value_counts().sort_index().to_string())

CODESTAB - Estabelecimento de Saúde
Total de estabelecimentos distintos: 26537
Nulos: 4777244
Não nulos: 12968907 (73.1%)

Top 10 estabelecimentos:
CODESTAB
0000001      1
0000010      1
0000011      2
0000013      2
0000014      6
0000017      2
0000018    690
0000019    587
0000020      1
0000021      9

Comprimento do código:
CODESTAB
7    12968907


## 14. ASSISTMED — Assistência Médica

| Código | Descrição |
|---|---|
| 1 | Sim |
| 2 (ou ausente) | Não / Ignorado |

In [23]:
# --- ASSISTMED ---
print("=" * 60)
print("ASSISTMED - Recebeu Assistência Médica")
print("=" * 60)

assistmed_counts = df_sim['ASSISTMED'].value_counts(dropna=False).sort_index()
print(assistmed_counts.to_string())
print(f"\nNulos: {df_sim['ASSISTMED'].isna().sum()}")
print(f"Valores distintos: {sorted(assistmed_counts.index[~pd.isna(assistmed_counts.index)].astype(str).tolist())}")

ASSISTMED - Recebeu Assistência Médica
ASSISTMED
1      9457796
2      1783703
9      1038055
NaN    5466597

Nulos: 5466597
Valores distintos: ['1', '2', '9']


## 15. APLICAÇÃO DO FILTRO DE MORTALIDADE MATERNA (REVISADO)

Validar o filtro revisado:

**Decisões:**
1. TPMORTEOCO 6 e 7 → normalizados para 9 (Ignorado)
2. SEXO removido da fato — usado apenas como critério de filtro
3. Critério único (sem exceção separada):

```sql
WHERE (SEXO_NORMALIZADO = 'F' AND IDADE BETWEEN '410' AND '449')
   OR (TPMORTEOCO_NORMALIZADO IN ('1', '2', '3', '4', '5'))
```

In [24]:
# --- APLICAR FILTRO DE MORTALIDADE MATERNA (REVISADO) ---
print("=" * 60)
print("FILTRO DE MORTALIDADE MATERNA (REVISADO)")
print("=" * 60)
print()
print("Decisões:")
print("  1. TPMORTEOCO 6 e 7 → normalizados para 9 (Ignorado)")
print("  2. SEXO removido da fato — usado apenas como critério de filtro")
print("  3. Critério único (sem exceção separada):")
print("     (SEXO_NORM = 'F' AND IDADE 10-49) OR (TPMORTEOCO_NORM IN 1-5)")
print()

# Normalizar TPMORTEOCO: 6 e 7 → 9 (Ignorado)
df_sim['tpmorteoco_normalizado'] = df_sim['TPMORTEOCO'].apply(
    lambda x: '9' if pd.notna(x) and str(x).strip() in ('6', '7') else x
)

print("--- TPMORTEOCO após normalização (6/7 → 9) ---")
tp_norm_counts = df_sim['tpmorteoco_normalizado'].value_counts(dropna=False).sort_index()
print(tp_norm_counts.to_string())

# Critério único (sem exceção separada, sem logging de outliers)
mask_filtro = (
    (df_sim['sexo_normalizado'] == 'F') &
    (df_sim['IDADE'].between('410', '449'))
) | (
    df_sim['tpmorteoco_normalizado'].isin(['1', '2', '3', '4', '5'])
)

total_geral = len(df_sim)
total_filtro = mask_filtro.sum()

print(f"\nTotal de registros no dataset completo: {total_geral:,}")
print(f"")
print(f"--- Filtro total: {total_filtro:,} ({total_filtro/total_geral*100:.1f}%)")
print(f"")

# Apenas uma curiosidade informativa, sem logging especial
print("--- Composição por SEXO normalizado ---")
print(df_sim[mask_filtro]['sexo_normalizado'].value_counts().to_string())

print(f"\nTotal de registros elegíveis para a fato: {total_filtro:,} ({total_filtro/total_geral*100:.1f}%)")

FILTRO DE MORTALIDADE MATERNA (REVISADO)

Decisões:
  1. TPMORTEOCO 6 e 7 → normalizados para 9 (Ignorado)
  2. SEXO removido da fato — usado apenas como critério de filtro
  3. Critério único (sem exceção separada):
     (SEXO_NORM = 'F' AND IDADE 10-49) OR (TPMORTEOCO_NORM IN 1-5)

--- TPMORTEOCO após normalização (6/7 → 9) ---
tpmorteoco_normalizado
1          7879
2          2243
3           906
4         14162
5          8075
8        898470
9        180664
NaN    16633752

Total de registros no dataset completo: 17,746,151

--- Filtro total: 867,420 (4.9%)

--- Composição por SEXO normalizado ---
sexo_normalizado
F    867408
I        11
M         1

Total de registros elegíveis para a fato: 867,420 (4.9%)


## 16. RESUMO — Valores Distintos por Variável

Tabela consolidada para validação do DDL.

In [25]:
# Resumo consolidado
print("=" * 80)
print("RESUMO - Valores Distintos por Variável para o Modelo Dimensional")
print("=" * 80)

variaveis = [
    ('SEXO', 'Atributo direto', df_sim['SEXO'].dropna().unique().tolist()),
    ('IDADE (1º dígito)', 'Dim_FaixaEtaria', df_sim['IDADE'].dropna().str[0].unique().tolist()),
    ('RACACOR', 'Dim_RacaCor', sorted(df_sim['RACACOR'].dropna().unique())),
    ('LOCOCOR', 'Dim_LocalOcorrencia', sorted(df_sim['LOCOCOR'].dropna().unique())),
    ('TPMORTEOCO', 'Dim_SituacaoGestacional', sorted(df_sim['TPMORTEOCO'].dropna().unique())),
    ('PARTO', 'Dim_TipoParto', sorted(df_sim['PARTO'].dropna().unique())),
    ('OBITOPARTO', 'Dim_MomentoObitoParto', sorted(df_sim['OBITOPARTO'].dropna().unique())),
    ('GRAVIDEZ', 'Dim_TipoGravidez', sorted(df_sim['GRAVIDEZ'].dropna().unique())),
    ('SEMAGESTAC', 'Dim_SemanaGestacao', sorted(df_sim['SEMAGESTAC'].dropna().unique())),
    ('ASSISTMED', 'Atributo direto', sorted(df_sim['ASSISTMED'].dropna().unique())),
]

for var, dim, valores in variaveis:
    nulos = df_sim[var].isna().sum() if var in df_sim.columns else 0
    print(f"\n{var:20s} → {dim:25s} | Nulos: {nulos:>8,} | Distintos: {len(valores):>4} | {valores}")

print(f"\n\n--- CID ---")
for col in ['CAUSABAS', 'CAUSAMAT', 'CAUSABAS_O']:
    n_cids = df_sim[col].nunique(dropna=False)
    nulos = df_sim[col].isna().sum()
    print(f"{col:20s} → Dim_CID (role-playing) | Nulos: {nulos:>8,} | CIDs distintos: {n_cids:>6,}")

print(f"\n--- Municípios ---")
for col in ['CODMUNOCOR', 'CODMUNRES']:
    n_muns = df_sim[col].nunique(dropna=False)
    nulos = df_sim[col].isna().sum()
    print(f"{col:20s} → Dim_Municipio (role-playing) | Nulos: {nulos:>8,} | Municípios distintos: {n_muns:>6,}")

print(f"\n--- Estabelecimentos ---")
n_estab = df_sim['CODESTAB'].nunique(dropna=False)
nulos_estab = df_sim['CODESTAB'].isna().sum()
print(f"{'CODESTAB':20s} → Dim_EstabelecimentoSaude       | Nulos: {nulos_estab:>8,} | Estabelecimentos distintos: {n_estab:>6,}")

print(f"\n--- Datas ---")
# DTOBITO no formato DDMMYYYY - extrair ano das posições 4-7
anos_dtobito = sorted(df_sim['DTOBITO'].dropna().str[4:8].unique())
print(f"{'DTOBITO':20s} → Dim_Tempo                       | Nulos: {df_sim['DTOBITO'].isna().sum():>8,} | Datas distintas: {df_sim['DTOBITO'].nunique():>6,} | Anos: {anos_dtobito}")

RESUMO - Valores Distintos por Variável para o Modelo Dimensional

SEXO                 → Atributo direto           | Nulos:        0 | Distintos:    3 | ['1', '2', '0']

IDADE (1º dígito)    → Dim_FaixaEtaria           | Nulos:        0 | Distintos:    7 | ['4', '2', '5', '0', '1', '3', '9']

RACACOR              → Dim_RacaCor               | Nulos:  463,028 | Distintos:    6 | ['1', '2', '3', '4', '5', '9']

LOCOCOR              → Dim_LocalOcorrencia       | Nulos:        3 | Distintos:    7 | ['1', '2', '3', '4', '5', '6', '9']

TPMORTEOCO           → Dim_SituacaoGestacional   | Nulos: 16,633,752 | Distintos:    9 | ['1', '2', '3', '4', '5', '6', '7', '8', '9']

PARTO                → Dim_TipoParto             | Nulos: 17,361,214 | Distintos:    3 | ['1', '2', '9']

OBITOPARTO           → Dim_MomentoObitoParto     | Nulos: 17,364,454 | Distintos:    4 | ['1', '2', '3', '9']

GRAVIDEZ             → Dim_TipoGravidez          | Nulos: 17,359,243 | Distintos:    4 | ['1', '2', '3', '9']

In [26]:
# ============================================================================
# RESUMO DO SUBCONJUNTO FILTRADO (registros que entrarão na tabela-fato)
# ============================================================================

# Aplicar o filtro de mortalidade materna (mask_filtro definido na seção 15)
df_filtrado = df_sim[mask_filtro].copy()
print("=" * 80)
print(f"RESUMO - Valores Distintos para o SUBCONJUNTO FILTRADO ({len(df_filtrado):,} registros)")
print("=" * 80)

# Classificar faixa etária apenas para registros com unidade=4 (anos)
def classificar_faixa(idade):
    if pd.isna(idade):
        return None
    for min_id, max_id, nome in faixas:
        if min_id <= idade <= max_id:
            return nome
    return 'DESCONHECIDA'

df_filtrado['idade_valor'] = df_filtrado['IDADE'].apply(
    lambda x: int(x[1:3]) if pd.notna(x) and str(x)[0] == '4' else None
)
df_filtrado['faixa_etaria'] = df_filtrado['idade_valor'].apply(classificar_faixa)

variaveis_filt = [
    ('SEXO (raw)', 'Filtro (não persiste)', df_filtrado['SEXO'].dropna().unique().tolist()),
    ('SEXO norm', 'Filtro (não persiste)', df_filtrado['sexo_normalizado'].dropna().unique().tolist()),
    ('FAIXA ETÁRIA', 'Dim_FaixaEtaria', df_filtrado['faixa_etaria'].dropna().unique().tolist()),
    ('RACACOR', 'Dim_RacaCor', sorted(df_filtrado['RACACOR'].dropna().unique())),
    ('LOCOCOR', 'Dim_LocalOcorrencia', sorted(df_filtrado['LOCOCOR'].dropna().unique())),
    ('TPMORTEOCO (norm)', 'Dim_SituacaoGestacional', sorted(df_filtrado['tpmorteoco_normalizado'].dropna().unique())),
    ('PARTO', 'Dim_TipoParto', sorted(df_filtrado['PARTO'].dropna().unique())),
    ('OBITOPARTO', 'Dim_MomentoObitoParto', sorted(df_filtrado['OBITOPARTO'].dropna().unique())),
    ('GRAVIDEZ', 'Dim_TipoGravidez', sorted(df_filtrado['GRAVIDEZ'].dropna().unique())),
    ('SEMAGESTAC', 'Dim_SemanaGestacao', sorted(df_filtrado['SEMAGESTAC'].dropna().unique())),
    ('ASSISTMED', 'Atributo direto', sorted(df_filtrado['ASSISTMED'].dropna().unique())),
]

for var, dim, valores in variaveis_filt:
    nulos = df_filtrado[var].isna().sum() if var in df_filtrado.columns else 0
    print(f"\n{var:20s} → {dim:28s} | Nulos: {nulos:>7,} | Distintos: {len(valores):>4} | {valores}")

print(f"\n\n--- CID ---")
for col in ['CAUSABAS', 'CAUSAMAT', 'CAUSABAS_O']:
    n_cids = df_filtrado[col].nunique(dropna=False)
    nulos = df_filtrado[col].isna().sum()
    print(f"{col:20s} → Dim_CID (role-playing) | Nulos: {nulos:>7,} | CIDs distintos: {n_cids:>6,}")

print(f"\n--- Municípios ---")
for col in ['CODMUNOCOR', 'CODMUNRES']:
    n_muns = df_filtrado[col].nunique(dropna=False)
    nulos = df_filtrado[col].isna().sum()
    print(f"{col:20s} → Dim_Municipio (role-playing) | Nulos: {nulos:>7,} | Municípios distintos: {n_muns:>6,}")

print(f"\n--- Estabelecimentos ---")
n_estab_f = df_filtrado['CODESTAB'].nunique(dropna=False)
nulos_estab_f = df_filtrado['CODESTAB'].isna().sum()
print(f"{'CODESTAB':20s} → Dim_EstabelecimentoSaude       | Nulos: {nulos_estab_f:>7,} | Estabelecimentos distintos: {n_estab_f:>6,}")

print(f"\n--- Datas ---")
anos_dtobito_f = sorted(df_filtrado['DTOBITO'].dropna().str[4:8].unique())
print(f"{'DTOBITO':20s} → Dim_Tempo                       | Nulos: {df_filtrado['DTOBITO'].isna().sum():>7,} | Datas distintas: {df_filtrado['DTOBITO'].nunique():>6,} | Anos: {anos_dtobito_f}")

print(f"\n--- Distribuição de faixa etária (subconjunto filtrado) ---")
print(df_filtrado['faixa_etaria'].value_counts().to_string())

print(f"\n--- Distribuição de TPMORTEOCO normalizado (subconjunto filtrado) ---")
print(df_filtrado['tpmorteoco_normalizado'].value_counts().to_string())

RESUMO - Valores Distintos para o SUBCONJUNTO FILTRADO (867,420 registros)

SEXO (raw)           → Filtro (não persiste)        | Nulos:       0 | Distintos:    3 | ['2', '0', '1']

SEXO norm            → Filtro (não persiste)        | Nulos:       0 | Distintos:    3 | ['F', 'I', 'M']

FAIXA ETÁRIA         → Dim_FaixaEtaria              | Nulos:       0 | Distintos:   10 | ['45-49', '40-44', '25-29', '30-34', '15-19', '10-14', '35-39', '20-24', '50+', 'DESCONHECIDA']

RACACOR              → Dim_RacaCor                  | Nulos:  20,739 | Distintos:    5 | ['1', '2', '3', '4', '5']

LOCOCOR              → Dim_LocalOcorrencia          | Nulos:       0 | Distintos:    7 | ['1', '2', '3', '4', '5', '6', '9']

TPMORTEOCO (norm)    → Dim_SituacaoGestacional      | Nulos:       0 | Distintos:    7 | ['1', '2', '3', '4', '5', '8', '9']

PARTO                → Dim_TipoParto                | Nulos: 866,523 | Distintos:    3 | ['1', '2', '9']

OBITOPARTO           → Dim_MomentoObitoParto        

In [27]:
# Consolidado final: verificar compatibilidade dos valores com o DDL (REVISADO)
print("=" * 80)
print("VALIDAÇÃO DE COMPATIBILIDADE COM O DDL (REVISADO)")
print("=" * 80)
print()
print("Mudanças desde v1:")
print("  - TPMORTEOCO 6/7 → normalizados para 9")
print("  - SEXO removido da fato (usado apenas para filtrar)")
print("  - RACACOR inclui código 9 (Ignorado)")
print()

# Usar TPMORTEOCO normalizado para validação
validacoes = [
    ("SEXO (filtro)", "Deve conter apenas {'0','1','2'} no raw SIM", 
     set(df_sim['SEXO'].dropna().unique()) <= {'0', '1', '2'}),
    ("RACACOR", "Deve conter apenas {'1','2','3','4','5','9', nulo}",
     set(df_sim['RACACOR'].dropna().unique()) <= {'1', '2', '3', '4', '5', '9'}),
    ("LOCOCOR", "Deve conter apenas {'1','2','3','4','5','6','9', nulo}",
     set(df_sim['LOCOCOR'].dropna().unique()) <= {'1', '2', '3', '4', '5', '6', '9'}),
    ("TPMORTEOCO (norm)", "Deve conter apenas {'1','2','3','4','5','8','9', nulo} (6/7→9)",
     set(df_sim['tpmorteoco_normalizado'].dropna().unique()) <= {'1', '2', '3', '4', '5', '8', '9'}),
    ("PARTO", "Deve conter apenas {'1','2','9', nulo}",
     set(df_sim['PARTO'].dropna().unique()) <= {'1', '2', '9'}),
    ("OBITOPARTO", "Deve conter apenas {'1','2','3','9', nulo}",
     set(df_sim['OBITOPARTO'].dropna().unique()) <= {'1', '2', '3', '9'}),
    ("GRAVIDEZ", "Deve conter apenas {'1','2','3','9', nulo}",
     set(df_sim['GRAVIDEZ'].dropna().unique()) <= {'1', '2', '3', '9'}),
    ("CAUSABAS", "Não deve conter nulos (NOT NULL na fato)",
     df_sim['CAUSABAS'].notna().all()),
    ("CODMUNOCOR", "Não deve conter nulos (NOT NULL na fato)",
     df_sim['CODMUNOCOR'].notna().all()),
]

print(f"{'Variável':25s} {'Esperado':<50s} {'Status':<10s}")
print("-" * 85)
for var, esperado, resultado in validacoes:
    status = "✓ OK" if resultado else "✗ PROBLEMA"
    print(f"{var:25s} {esperado:<50s} {status:<10s}")

VALIDAÇÃO DE COMPATIBILIDADE COM O DDL (REVISADO)

Mudanças desde v1:
  - TPMORTEOCO 6/7 → normalizados para 9
  - SEXO removido da fato (usado apenas para filtrar)
  - RACACOR inclui código 9 (Ignorado)

Variável                  Esperado                                           Status    
-------------------------------------------------------------------------------------
SEXO (filtro)             Deve conter apenas {'0','1','2'} no raw SIM        ✓ OK      
RACACOR                   Deve conter apenas {'1','2','3','4','5','9', nulo} ✓ OK      
LOCOCOR                   Deve conter apenas {'1','2','3','4','5','6','9', nulo} ✓ OK      
TPMORTEOCO (norm)         Deve conter apenas {'1','2','3','4','5','8','9', nulo} (6/7→9) ✓ OK      
PARTO                     Deve conter apenas {'1','2','9', nulo}             ✓ OK      
OBITOPARTO                Deve conter apenas {'1','2','3','9', nulo}         ✓ OK      
GRAVIDEZ                  Deve conter apenas {'1','2','3','9', nulo}         

In [28]:
df_filtrado.OBITOPARTO.value_counts()

OBITOPARTO
3    865
9     21
Name: count, dtype: int64

In [29]:
# ============================================================================
# DATASET: OBITOPARTO não nulo E fora do filtro (df_filtrado)
# Registros com OBITOPARTO preenchido que NÃO entraram na fato
# ============================================================================

mask_obitoparto_nao_nulo = df_sim['OBITOPARTO'].notna()
mask_fora_filtro = ~mask_filtro

df_obitoparto_fora_filtro = df_sim[
    mask_obitoparto_nao_nulo & mask_fora_filtro
].copy()

print("=" * 80)
print(f"DATASET: OBITOPARTO não nulo E fora do filtro")
print(f"Total de registros: {len(df_obitoparto_fora_filtro):,}")
print("=" * 80)

print("\n--- Distribuição de OBITOPARTO ---")
print(df_obitoparto_fora_filtro['OBITOPARTO'].value_counts(dropna=False).to_string())

print("\n--- Distribuição por SEXO (raw) ---")
print(df_obitoparto_fora_filtro['SEXO'].value_counts(dropna=False).to_string())

print("\n--- Distribuição por SEXO normalizado ---")
print(df_obitoparto_fora_filtro['sexo_normalizado'].value_counts(dropna=False).to_string())

print("\n--- Cruzamento OBITOPARTO × TPMORTEOCO ---")
print(pd.crosstab(df_obitoparto_fora_filtro['OBITOPARTO'],
                  df_obitoparto_fora_filtro['tpmorteoco_normalizado'],
                  margins=True).to_string())

# Por que não entraram no filtro? Verificar IDADE
print("\n--- Cruzamento OBITOPARTO × IDADE (1º dígito) ---")
df_obitoparto_fora_filtro['idade_prefixo'] = df_obitoparto_fora_filtro['IDADE'].str[0]
print(pd.crosstab(df_obitoparto_fora_filtro['OBITOPARTO'],
                  df_obitoparto_fora_filtro['idade_prefixo'],
                  margins=True).to_string())

# Salvar dataset (descomentar para exportar)
# df_obitoparto_fora_filtro.to_csv("../arquivos/obitoparto_fora_filtro.csv",
#                                  sep=';', index=False, encoding='utf-8')

DATASET: OBITOPARTO não nulo E fora do filtro
Total de registros: 380,811

--- Distribuição de OBITOPARTO ---
OBITOPARTO
3    372390
9      8408
2         7
1         6

--- Distribuição por SEXO (raw) ---
SEXO
1    210307
2    168329
0      2175

--- Distribuição por SEXO normalizado ---
sexo_normalizado
M    210307
F    168329
I      2175

--- Cruzamento OBITOPARTO × TPMORTEOCO ---
tpmorteoco_normalizado      8     9    All
OBITOPARTO                                
3                       14333  3508  17841
9                         141   117    258
All                     14474  3625  18099

--- Cruzamento OBITOPARTO × IDADE (1º dígito) ---
idade_prefixo      0      1       2       3   4   9     All
OBITOPARTO                                                 
1                  1      0       0       0   3   2       6
2                  0      0       0       0   0   7       7
3              31070  61510  181671   98128   0  11  372390
9                267    396    2101    5603   9

## 17. Decomposição do Filtro — Quantos registros entram por cada critério

O `df_filtrado` é a **união** de dois critérios:

1. **Filtro SEXO+IDADE** (`mask_sexo_idade`): `SEXO_NORM = 'F'` **E** `IDADE` entre `410` e `449` (10–49 anos)
2. **Filtro TPMORTEOCO** (`mask_tpmorteoco`): `TPMORTEOCO_NORM` em `{1, 2, 3, 4, 5}` (marcador de óbito materno)

Para saber quantos registros entram *por conta de cada filtro*, decompomos a união em 3 conjuntos **disjuntos**:

- **Só SEXO+IDADE**: passa no 1º e **NÃO** passa no 2º
- **Só TPMORTEOCO**: passa no 2º e **NÃO** passa no 1º
- **Ambos**: passa nos dois

Total (união = `df_filtrado`) = Só SEXO+IDADE + Só TPMORTEOCO + Ambos


In [30]:
# ============================================================================
# DECOMPOSIÇÃO DO FILTRO — QUANTOS REGISTROS ENTRAM POR CADA CRITÉRIO
# ============================================================================

# Critério 1: SEXO + IDADE (feminino, 10-49 anos)
mask_sexo_idade = (
    (df_sim['sexo_normalizado'] == 'F') &
    (df_sim['IDADE'].between('410', '449'))
)

# Critério 2: TPMORTEOCO (situação gestacional do óbito 1-5)
mask_tpmorteoco = df_sim['tpmorteoco_normalizado'].isin(['1', '2', '3', '4', '5'])

# Conjuntos disjuntos
n_so_sexo_idade = (mask_sexo_idade & ~mask_tpmorteoco).sum()   # só critério 1
n_so_tpmorteoco = (mask_tpmorteoco & ~mask_sexo_idade).sum()   # só critério 2
n_ambos         = (mask_sexo_idade &  mask_tpmorteoco).sum()   # ambos
n_union         = (mask_sexo_idade |  mask_tpmorteoco).sum()   # união = df_filtrado

print("=" * 80)
print("DECOMPOSIÇÃO DO FILTRO DE MORTALIDADE MATERNA")
print("=" * 80)
print(f"Total SIM (df_sim):            {len(df_sim):>10,}")
print()
print(f"Critério 1 — SEXO+IDADE (F 10-49):   {mask_sexo_idade.sum():>10,}   (entram por sexo+idade, com ou sem TPMORTEOCO)")
print(f"Critério 2 — TPMORTEOCO (1-5):       {mask_tpmorteoco.sum():>10,}   (entram por TPMORTEOCO, com ou sem sexo+idade)")
print()
print("Decomposição em conjuntos disjuntos:")
print(f"  Só SEXO+IDADE (não tem TPMORTEOCO 1-5):  {n_so_sexo_idade:>10,}")
print(f"  Só TPMORTEOCO (não é F 10-49):           {n_so_tpmorteoco:>10,}")
print(f"  Ambos (F 10-49 E TPMORTEOCO 1-5):        {n_ambos:>10,}")
print(f"  {'-' * 62}")
print(f"  UNIÃO (df_filtrado):                     {n_union:>10,}")
print()
print(f"Conferência: mask_filtro.sum() = {mask_filtro.sum():,} | len(df_filtrado) = {len(df_filtrado):,}")
print(f"  Só SEXO+IDADE + Só TPMORTEOCO + Ambos = {n_so_sexo_idade + n_so_tpmorteoco + n_ambos:,}")
print()
print("Percentuais sobre o total filtrado:")
print(f"  Só SEXO+IDADE: {n_so_sexo_idade/n_union*100:5.1f}%")
print(f"  Só TPMORTEOCO: {n_so_tpmorteoco/n_union*100:5.1f}%")
print(f"  Ambos:         {n_ambos/n_union*100:5.1f}%")


DECOMPOSIÇÃO DO FILTRO DE MORTALIDADE MATERNA
Total SIM (df_sim):            17,746,151

Critério 1 — SEXO+IDADE (F 10-49):      864,542   (entram por sexo+idade, com ou sem TPMORTEOCO)
Critério 2 — TPMORTEOCO (1-5):           33,265   (entram por TPMORTEOCO, com ou sem sexo+idade)

Decomposição em conjuntos disjuntos:
  Só SEXO+IDADE (não tem TPMORTEOCO 1-5):     834,155
  Só TPMORTEOCO (não é F 10-49):                2,878
  Ambos (F 10-49 E TPMORTEOCO 1-5):            30,387
  --------------------------------------------------------------
  UNIÃO (df_filtrado):                        867,420

Conferência: mask_filtro.sum() = 867,420 | len(df_filtrado) = 867,420
  Só SEXO+IDADE + Só TPMORTEOCO + Ambos = 867,420

Percentuais sobre o total filtrado:
  Só SEXO+IDADE:  96.2%
  Só TPMORTEOCO:   0.3%
  Ambos:           3.5%


## 18. Dataset específico — Apenas o filtro de TPMORTEOCO

Cria `df_tpmorteoco`: **todos** os registros do SIM cujo `TPMORTEOCO_NORM ∈ {1, 2, 3, 4, 5}`, independentemente de SEXO/IDADE.

Aqui também quantificamos quantos desses registros **só** entram por TPMORTEOCO (i.e., não seriam capturados pelo critério SEXO+IDADE) e o motivo (sexo ou idade fora da faixa).


In [31]:
# ============================================================================
# DATASET ESPECÍFICO: APENAS O FILTRO DE TPMORTEOCO
# ============================================================================

# Redefinir as máscaras para o cell ficar autocontido
mask_sexo_idade = (
    (df_sim['sexo_normalizado'] == 'F') &
    (df_sim['IDADE'].between('410', '449'))
)
mask_tpmorteoco = df_sim['tpmorteoco_normalizado'].isin(['1', '2', '3', '4', '5'])

# Dataset específico do filtro TPMORTEOCO
df_tpmorteoco = df_sim[mask_tpmorteoco].copy()

print("=" * 80)
print(f"DATASET TPMORTEOCO (filtro apenas por TPMORTEOCO 1-5)")
print(f"Registros: {len(df_tpmorteoco):,}")
print("=" * 80)

print("\n--- Distribuição de TPMORTEOCO (normalizado) ---")
print(df_tpmorteoco['tpmorteoco_normalizado'].value_counts(dropna=False).sort_index().to_string())

print("\n--- Distribuição por SEXO (normalizado) ---")
print(df_tpmorteoco['sexo_normalizado'].value_counts(dropna=False).to_string())

print("\n--- Cruzamento TPMORTEOCO × SEXO normalizado ---")
print(pd.crosstab(df_tpmorteoco['tpmorteoco_normalizado'],
                  df_tpmorteoco['sexo_normalizado'], margins=True).to_string())

# Quantos do dataset TPMORTEOCO NÃO passariam no critério SEXO+IDADE?
dentro = df_tpmorteoco.copy()
dentro['passa_sexo_idade'] = (
    (dentro['sexo_normalizado'] == 'F') &
    dentro['IDADE'].between('410', '449')
)
n_so_tpmorteoco = (~dentro['passa_sexo_idade']).sum()
n_ambos = dentro['passa_sexo_idade'].sum()

print(f"\nDos {len(df_tpmorteoco):,} registros que entram por TPMORTEOCO:")
print(f"  - Também passam no SEXO+IDADE (Ambos):        {n_ambos:>10,}")
print(f"  - SÓ entram por TPMORTEOCO (não é F 10-49):   {n_so_tpmorteoco:>10,}")

# Motivo de não passar no SEXO+IDADE (sexo × unidade de medida da idade)
dentro['idade_prefixo'] = dentro['IDADE'].str[0]
print("\n--- Por que não passam no SEXO+IDADE? (SEXO × 1º dígito do IDADE) ---")
print(pd.crosstab(dentro['sexo_normalizado'], dentro['idade_prefixo'], margins=True).to_string())

# Opcional: exportar o dataset (descomente para salvar)
# df_tpmorteoco.to_csv("../arquivos/sim_filtro_tpmorteoco.csv",
#                      sep=';', index=False, encoding='utf-8')


DATASET TPMORTEOCO (filtro apenas por TPMORTEOCO 1-5)
Registros: 33,265

--- Distribuição de TPMORTEOCO (normalizado) ---
tpmorteoco_normalizado
1     7879
2     2243
3      906
4    14162
5     8075

--- Distribuição por SEXO (normalizado) ---
sexo_normalizado
F    33253
I       11
M        1

--- Cruzamento TPMORTEOCO × SEXO normalizado ---
sexo_normalizado            F   I  M    All
tpmorteoco_normalizado                     
1                        7875   3  1   7879
2                        2240   3  0   2243
3                         904   2  0    906
4                       14159   3  0  14162
5                        8075   0  0   8075
All                     33253  11  1  33265

Dos 33,265 registros que entram por TPMORTEOCO:
  - Também passam no SEXO+IDADE (Ambos):            30,387
  - SÓ entram por TPMORTEOCO (não é F 10-49):        2,878

--- Por que não passam no SEXO+IDADE? (SEXO × 1º dígito do IDADE) ---
idade_prefixo       0    1    2    3      4   9    All
sexo_norma

## 19. Passo 2 — Mulheres com idade em anos (prefixo `4`) fora de 10–49, no grupo "só TPMORTEOCO"

Dentro do grupo **só TPMORTEOCO** (2.442), havia **1.578 mulheres com `IDADE` medida em anos (prefixo `4`) mas fora da faixa 410–449**. Aqui detalhamos essas 1.578: quantas têm **< 10 anos** vs **≥ 50 anos** (e a distribuição por idade exata), para decidir se são óbitos maternos legítimos ou erros de codificação.


In [32]:
# ============================================================================
# PASSO 2 — MULHERES PREFIXO 4 FORA DE 10-49 NO GRUPO "SÓ TPMORTEOCO"
# ============================================================================

# Reconstruir máscaras (cell autocontido)
mask_sexo_idade = (
    (df_sim['sexo_normalizado'] == 'F') &
    (df_sim['IDADE'].between('410', '449'))
)
mask_tpmorteoco = df_sim['tpmorteoco_normalizado'].isin(['1', '2', '3', '4', '5'])

# Grupo "só TPMORTEOCO" (passa no TPMORTEOCO, NÃO passa no SEXO+IDADE)
so_tp = df_sim[mask_tpmorteoco & ~mask_sexo_idade].copy()

# Mulheres com idade em anos (prefixo 4) fora de 10-49
mulheres_prefixo4 = so_tp[
    (so_tp['sexo_normalizado'] == 'F') &
    (so_tp['IDADE'].str[0] == '4')
].copy()
mulheres_prefixo4['idade_valor'] = mulheres_prefixo4['IDADE'].str[1:3].astype(int)

print("=" * 80)
print(f"PASSO 2 — Mulheres com idade em anos (prefixo 4) fora de 10-49")
print(f"Total: {len(mulheres_prefixo4):,}  (esperado 1.578)")
print("=" * 80)

# Quebra <10 vs 10-49 vs >=50
def classifica_fora_faixa(idade):
    if idade < 10:
        return '<10 anos'
    elif idade <= 49:
        return '10-49 (não deveria estar aqui)'
    else:
        return '>=50 anos'

mulheres_prefixo4['faixa_fora'] = mulheres_prefixo4['idade_valor'].apply(classifica_fora_faixa)

print("\n--- Quebra por faixa (fora do filtro 10-49) ---")
print(mulheres_prefixo4['faixa_fora'].value_counts().to_string())

print("\n--- Distribuição por idade exata ---")
print(mulheres_prefixo4['idade_valor'].value_counts().sort_index().to_string())

print("\n--- Cruzamento idade exata × TPMORTEOCO ---")
print(pd.crosstab(mulheres_prefixo4['idade_valor'],
                  mulheres_prefixo4['tpmorteoco_normalizado'],
                  margins=True).to_string())

print("\n--- Validação rápida: CAUSAMAT preenchida (sinal de causa materna) ---")
cm = mulheres_prefixo4['CAUSAMAT'].notna()
print(cm.value_counts().to_string())
print(f"Com CAUSAMAT: {cm.sum():,} ({cm.mean()*100:.1f}%) | Sem CAUSAMAT: {(~cm).sum():,}")

print("\n--- Amostra de registros <10 anos (primeiros 10) ---")
menores = mulheres_prefixo4[mulheres_prefixo4['faixa_fora'] == '<10 anos']
cols_mostrar = ['IDADE', 'idade_valor', 'TPMORTEOCO', 'tpmorteoco_normalizado',
                'CAUSABAS', 'CAUSAMAT', 'DTOBITO']
print(menores[cols_mostrar].head(10).to_string())


PASSO 2 — Mulheres com idade em anos (prefixo 4) fora de 10-49
Total: 1,906  (esperado 1.578)

--- Quebra por faixa (fora do filtro 10-49) ---
faixa_fora
>=50 anos    1898
<10 anos        8

--- Distribuição por idade exata ---
idade_valor
8       4
9       4
50    110
51    131
52    129
53    133
54    131
55     94
56     94
57    108
58     92
59    116
60    132
61    113
62    115
63    130
64    146
65    124

--- Cruzamento idade exata × TPMORTEOCO ---
tpmorteoco_normalizado     1    2   3   4    5   All
idade_valor                                         
8                          3    1   0   0    0     4
9                          2    1   0   0    1     4
50                        70   20   4   3   13   110
51                        88   19   4   9   11   131
52                        96   18   1   1   13   129
53                        97   20   3   1   12   133
54                        87   23   5   3   13   131
55                        68    8   4   2   12    94
56   

## 20. Análise dos registros com CAUSAMAT preenchido

**Hipótese levantada:** `CAUSAMAT` não estaria associado a óbitos maternos no sentido clássico, mas sim à **investigação de óbitos neonatais e fetais**.

Para avaliar, analisamos **todos os registros de `df_sim` com `CAUSAMAT` preenchido**, verificando:

- **Idade** (unidade de medida e faixa etária) — se a maioria tiver < 1 ano (prefixo 0–3), apoia a hipótese neonatal/fetal;
- **TPMORTEOCO** (incidência) — se a maioria for 8/9/ausente (não gestacional), reforça que não se trata de óbito materno;
- **CID em `CAUSAMAT`** — se é materno (O00–O99 / A34) ou outro;
- **Relação com o filtro de mortalidade materna** (`mask_filtro`).



In [33]:
# ============================================================================
# ANÁLISE: REGISTROS COM CAUSAMAT PREENCHIDO EM df_sim
# Hipótese: CAUSAMAT ligado à investigação de óbitos neonatais/fetais
# ============================================================================

mask_causamat = df_sim['CAUSAMAT'].notna()
df_causamat = df_sim[mask_causamat].copy()

print("=" * 80)
print("REGISTROS COM CAUSAMAT PREENCHIDO")
print(f"Total: {len(df_causamat):,} de {len(df_sim):,} ({len(df_causamat)/len(df_sim)*100:.2f}%)")
print("=" * 80)

# --- 1) SEXO ---
print("\n--- 1) SEXO (normalizado) ---")
print(df_causamat['sexo_normalizado'].value_counts(dropna=False).to_string())

# --- 2) IDADE: unidade de medida (1º dígito) ---
UNID_IDADE = {
    '0': '0 = minutos', '1': '1 = horas', '2': '2 = dias',
    '3': '3 = meses', '4': '4 = anos', '5': '5 = 100+ anos', '9': '9 = ignorada',
}
df_causamat['idade_prefixo'] = df_causamat['IDADE'].str[0]
print("\n--- 2) IDADE — unidade de medida (1º dígito) ---")
print(df_causamat['idade_prefixo'].map(UNID_IDADE).value_counts(dropna=False).to_string())

# --- 3) TPMORTEOCO (normalizado) — incidência ---
print("\n--- 3) TPMORTEOCO (normalizado) — incidência ---")
tp_cm = df_causamat['tpmorteoco_normalizado'].value_counts(dropna=False).sort_index()
print(tp_cm.to_string())
print(f"\nTPMORTEOCO 1-5 (gestacional): {tp_cm.get('1',0)+tp_cm.get('2',0)+tp_cm.get('3',0)+tp_cm.get('4',0)+tp_cm.get('5',0):,} "
      f"({(tp_cm.get('1',0)+tp_cm.get('2',0)+tp_cm.get('3',0)+tp_cm.get('4',0)+tp_cm.get('5',0))/len(df_causamat)*100:.1f}%)")

# --- 4) Cruzamento TPMORTEOCO × unidade de idade ---
print("\n--- 4) Cruzamento TPMORTEOCO_norm × IDADE prefixo ---")
print(pd.crosstab(df_causamat['tpmorteoco_normalizado'],
                  df_causamat['idade_prefixo'], margins=True).to_string())

# --- 5) Distribuição de idade em anos (prefixo 4) ---
cm_anos = df_causamat[df_causamat['idade_prefixo'] == '4'].copy()
cm_anos['idade_valor'] = cm_anos['IDADE'].str[1:3].astype(int)

def faixa_idade(v):
    if v < 1: return '0'
    if v < 10: return '1-9'
    if v < 50: return '10-49'
    return '50+'

cm_anos['faixa'] = cm_anos['idade_valor'].apply(faixa_idade)
print("\n--- 5) Faixa etária (apenas idade em anos, prefixo 4) ---")
print(cm_anos['faixa'].value_counts().sort_index().to_string())
print("\nTop 15 idades exatas:")
print(cm_anos['idade_valor'].value_counts().head(15).to_string())

# --- 6) CAUSAMAT é CID materno? (O00-O99 / A34) ---
def eh_cid_materno(cid):
    c = str(cid).strip().upper()
    return c.startswith('O') or c.startswith('A34')

df_causamat['cid_materno'] = df_causamat['CAUSAMAT'].apply(eh_cid_materno)
print("\n--- 6) CAUSAMAT contém CID materno (O00-O99 / A34)? ---")
print(df_causamat['cid_materno'].value_counts().to_string())

print("\n--- Cruzamento CID materno (sim/não) × IDADE prefixo ---")
print(pd.crosstab(df_causamat['cid_materno'], df_causamat['idade_prefixo'],
                  margins=True).to_string())

print("\n--- Top 15 CIDs em CAUSAMAT ---")
print(df_causamat['CAUSAMAT'].value_counts().head(15).to_string())

# --- 7) Relação com o filtro de mortalidade materna ---
n_cm_filtro = (mask_causamat & mask_filtro).sum()
n_cm_fora = (mask_causamat & ~mask_filtro).sum()
print(f"\n--- 7) Relação com o filtro de mortalidade materna (mask_filtro) ---")
print(f"Com CAUSAMAT e DENTRO do filtro: {n_cm_filtro:,} ({n_cm_filtro/len(df_causamat)*100:.1f}%)")
print(f"Com CAUSAMAT e FORA do filtro:   {n_cm_fora:,} ({n_cm_fora/len(df_causamat)*100:.1f}%)")


REGISTROS COM CAUSAMAT PREENCHIDO
Total: 335 de 17,746,151 (0.00%)

--- 1) SEXO (normalizado) ---
sexo_normalizado
F    335

--- 2) IDADE — unidade de medida (1º dígito) ---
idade_prefixo
4 = anos    335

--- 3) TPMORTEOCO (normalizado) — incidência ---
tpmorteoco_normalizado
1      207
2        2
3        3
4       36
5       66
8        8
9        1
NaN     12

TPMORTEOCO 1-5 (gestacional): 314 (93.7%)

--- 4) Cruzamento TPMORTEOCO_norm × IDADE prefixo ---
idade_prefixo             4  All
tpmorteoco_normalizado          
1                       207  207
2                         2    2
3                         3    3
4                        36   36
5                        66   66
8                         8    8
9                         1    1
All                     323  323

--- 5) Faixa etária (apenas idade em anos, prefixo 4) ---
faixa
10-49    335

Top 15 idades exatas:
idade_valor
21    25
27    19
24    19
26    17
31    17
19    17
30    17
18    16
25    15
22    15
16  

## 21. FILTRO V2 — Fato = universo de mulheres em idade fértil com possível óbito de causa materna

**Decisão consolidada:** manter na fato os **1.578 registros** com `TPMORTEOCO` 1–5 e idade em anos (prefixo `4`) entre **8 e 65 anos**.

**Novo critério do filtro (V2):**

```sql
(SEXO_NORM = 'F' AND IDADE BETWEEN '410' AND '449')                             -- mulheres 10–49 anos
OR
(TPMORTEOCO_NORM IN ('1','2','3','4','5') AND IDADE BETWEEN '408' AND '465')    -- marcador gestacional, 8–65 anos
```

**Racional:** a fato deve conter **todas as mulheres em idade fértil que podem ter morrido de causa materna, independentemente da causa básica**, pois os CIDs podem mascarar a causa real do óbito. Os ~700 mil registros resultantes formam o **universo de mortes a investigar** daqui pra frente.

**O que muda vs V1:** o critério `TPMORTEOCO` passa a **exigir `IDADE` entre 8 e 65 anos** (prefixo `4`), o que mantém os 1.578 registros (idades 8–65) e **exclui** da captura por TPMORTEOCO os casos com idade < 1 ano (prefixo 0–3), idade ignorada (prefixo 9) e idade fora de 8–65.


In [34]:
# ============================================================================
# FILTRO V2 — UNIVERSO DE MORTES MATERNAS A INVESTIGAR (df_filtrado_v2)
# Critério: (SEXO='F' e IDADE 10-49) OU (TPMORTEOCO 1-5 e IDADE 8-65)
# ============================================================================

# --- V1 (reconstruído, autocontido) para comparação ---
mask_sexo_idade_v1 = (
    (df_sim['sexo_normalizado'] == 'F') &
    (df_sim['IDADE'].between('410', '449'))
)
mask_tp_v1 = df_sim['tpmorteoco_normalizado'].isin(['1', '2', '3', '4', '5'])
mask_filtro_v1 = mask_sexo_idade_v1 | mask_tp_v1

# --- V2 ---
mask_sexo_idade_v2 = mask_sexo_idade_v1.copy()          # critério 1 inalterado
mask_tp_v2 = (
    mask_tp_v1 &
    (df_sim['IDADE'].between('408', '465'))              # idade 8-65 anos (prefixo 4)
)
mask_filtro_v2 = mask_sexo_idade_v2 | mask_tp_v2
df_filtrado_v2 = df_sim[mask_filtro_v2].copy()

print("=" * 80)
print("FILTRO V2 — UNIVERSO DE MORTES MATERNAS A INVESTIGAR")
print("=" * 80)
print(f"Total SIM (df_sim):              {len(df_sim):>10,}")
print()
print(f"Critério 1 — SEXO+IDADE (F 10-49):       {mask_sexo_idade_v2.sum():>10,}")
print(f"Critério 2 — TPMORTEOCO 1-5 E 8-65 a:     {mask_tp_v2.sum():>10,}")
print()

# Decomposição disjunta (V2)
n_so_sexo = (mask_sexo_idade_v2 & ~mask_tp_v2).sum()
n_so_tp   = (mask_tp_v2 & ~mask_sexo_idade_v2).sum()
n_ambos   = (mask_sexo_idade_v2 & mask_tp_v2).sum()
n_union   = mask_filtro_v2.sum()

print("Decomposição (V2):")
print(f"  Só SEXO+IDADE (F 10-49):          {n_so_sexo:>10,}")
print(f"  Só TPMORTEOCO (8-65 anos):        {n_so_tp:>10,}")
print(f"  Ambos:                            {n_ambos:>10,}")
print(f"  {'-' * 60}")
print(f"  UNIÃO (df_filtrado_v2):           {n_union:>10,}")
print()

# Comparação V1 -> V2
print("--- Comparação V1 → V2 ---")
print(f"  V1 (df_filtrado):        {len(df_filtrado):>10,}")
print(f"  V2 (df_filtrado_v2):     {n_union:>10,}")
print(f"  Removidos (V1 - V2):     {len(df_filtrado) - n_union:>10,}")

# Validar que os 1.578 alvo (TPMORTEOCO 1-5, prefixo 4, 8-65 a) estão todos no V2
alvo = df_sim[
    mask_tp_v1 &
    (df_sim['IDADE'].str[0] == '4') &
    df_sim['IDADE'].between('408', '465') &
    ~mask_sexo_idade_v1
]
dentro_v2 = mask_filtro_v2.loc[alvo.index].sum()
print()
print(f"--- Validação do alvo (esperado 1.578) ---")
print(f"  Registros alvo (TPMORTEOCO 1-5, 8-65 a, fora de 10-49): {len(alvo):,}")
print(f"  Desses, DENTRO do V2: {dentro_v2:,} | FORA: {len(alvo) - dentro_v2:,}")

# Perfil dos removidos na V2 (estavam no V1, saíram)
removidos = df_sim[mask_filtro_v1 & ~mask_filtro_v2]
print()
print(f"--- Perfil dos REMOVIDOS na V2 ({len(removidos):,}) ---")
print("  Por SEXO normalizado:")
print(removidos['sexo_normalizado'].value_counts(dropna=False).to_string())
print("  Por unidade de medida da IDADE (1º dígito):")
print(removidos['IDADE'].str[0].value_counts(dropna=False).sort_index().to_string())
print("  Por TPMORTEOCO (normalizado):")
print(removidos['tpmorteoco_normalizado'].value_counts(dropna=False).sort_index().to_string())

# Composição do df_filtrado_v2
print(f"\n--- Composição do df_filtrado_v2 ({len(df_filtrado_v2):,}) ---")
print("  Por SEXO normalizado:")
print(df_filtrado_v2['sexo_normalizado'].value_counts(dropna=False).to_string())
print("  Por TPMORTEOCO (normalizado):")
print(df_filtrado_v2['tpmorteoco_normalizado'].value_counts(dropna=False).sort_index().to_string())


FILTRO V2 — UNIVERSO DE MORTES MATERNAS A INVESTIGAR
Total SIM (df_sim):              17,746,151

Critério 1 — SEXO+IDADE (F 10-49):          864,542
Critério 2 — TPMORTEOCO 1-5 E 8-65 a:         32,294

Decomposição (V2):
  Só SEXO+IDADE (F 10-49):             834,155
  Só TPMORTEOCO (8-65 anos):             1,907
  Ambos:                                30,387
  ------------------------------------------------------------
  UNIÃO (df_filtrado_v2):              866,449

--- Comparação V1 → V2 ---
  V1 (df_filtrado):           867,420
  V2 (df_filtrado_v2):        866,449
  Removidos (V1 - V2):            971

--- Validação do alvo (esperado 1.578) ---
  Registros alvo (TPMORTEOCO 1-5, 8-65 a, fora de 10-49): 1,907
  Desses, DENTRO do V2: 1,907 | FORA: 0

--- Perfil dos REMOVIDOS na V2 (971) ---
  Por SEXO normalizado:
sexo_normalizado
F    960
I     10
M      1
  Por unidade de medida da IDADE (1º dígito):
IDADE
0    252
1    367
2    177
3    156
4      1
9     18
  Por TPMORTEOCO (no

### Novo Filtro a ser utilizado para os dados do SIM

WHERE (SEXO_NORMALIZADO = 'F' AND IDADE BETWEEN '410' AND '449') OR (TPMORTEOCO_NORMALIZADO IN ('1','2','3','4','5') AND IDADE BETWEEN '408' AND '465')

## Dataset MIF = Mortes de mulheres em idade fértil

In [35]:
mif = df_filtrado_v2[['ano_dtobito', 'data_iso', 'IDADE', 'RACACOR', 'LOCOCOR', 'tpmorteoco_normalizado', 'CAUSABAS', 'CAUSABAS_O', 'CODMUNOCOR', 'CODMUNRES', 'ASSISTMED', 'CODESTAB']].copy()

In [36]:
mif.to_csv("../arquivos/mortes_mulheres_idade_fertil.csv", sep=';', index=False, encoding='utf-8')

In [37]:
mif.head()

,ano_dtobito,data_iso,IDADE,RACACOR,LOCOCOR,tpmorteoco_normalizado,CAUSABAS,CAUSABAS_O,CODMUNOCOR,CODMUNRES,ASSISTMED,CODESTAB
27,2014,2014-01-03,449,1,1,8,I619,I619,120020,120020,1,5336171
37,2014,2014-01-04,441,2,1,8,N390,N390,120040,120040,1,2001578
60,2014,2014-01-08,449,2,1,8,C80,C80,120040,120040,1,2001586
77,2014,2014-01-09,426,4,5,8,I219,R99,120040,120040,1,NaN
96,2014,2014-01-12,444,4,1,8,C539,C539,120040,120040,1,2001586


In [38]:
cnes = pd.read_csv('../arquivos/CNES/cnes_estabelecimentos.csv', sep=';', decimal=',', encoding='latin1', low_memory=False)

In [39]:
# Verificação tratanto CODESTAB e CO_CNES como valores numéricos
codigo_estab_num = pd.to_numeric(mif['CODESTAB'], errors='coerce')
codigos_cnes_num = pd.to_numeric(cnes['CO_CNES'], errors='coerce')

mask_existencia_num = codigo_estab_num.isin(codigos_cnes_num.dropna())

mif['codigo_existe_no_cnes_num'] = mask_existencia_num

print(f"Total de registros em mif: {len(mif)}")
print(f"Registros com código presente no CNES (numérico): {mask_existencia_num.sum()}")
print(f"Registros sem correspondência (numérico): {(~mask_existencia_num).sum()}")

Total de registros em mif: 866449
Registros com código presente no CNES (numérico): 656834
Registros sem correspondência (numérico): 209615


In [40]:
# Cria chaves numéricas para o join
# Garante que as colunas tenham o mesmo tipo para o merge

mif_join = mif.copy()
cnes_join = cnes.copy()

mif_join['CODESTAB_num'] = pd.to_numeric(mif_join['CODESTAB'], errors='coerce')
cnes_join['CO_CNES_num'] = pd.to_numeric(cnes_join['CO_CNES'], errors='coerce')

# Remove colunas duplicadas/irrelevantes do CNES para evitar conflitos no merge
cnes_join = cnes_join.drop(columns=['CO_CNES'], errors='ignore')

# Left join
merged_df = (
    mif_join
    .merge(
        cnes_join,
        left_on='CODESTAB_num',
        right_on='CO_CNES_num',
        how='left',
        suffixes=('_mif', '_cnes')
    )
)

# Remove a chave auxiliar após o merge
merged_df = merged_df.drop(columns=['CODESTAB_num', 'CO_CNES_num'], errors='ignore')

print(f"Shape do mif: {mif.shape}")
print(f"Shape do merged_df: {merged_df.shape}")
print("Colunas adicionadas do CNES:")
print([col for col in merged_df.columns if col.endswith('_cnes') or col in cnes.columns])

Shape do mif: (866449, 13)
Shape do merged_df: (866449, 48)
Colunas adicionadas do CNES:
['CO_UNIDADE', 'CO_UF', 'CO_IBGE', 'NU_CNPJ_MANTENEDORA', 'NO_RAZAO_SOCIAL', 'NO_FANTASIA', 'CO_NATUREZA_ORGANIZACAO', 'DS_NATUREZA_ORGANIZACAO', 'TP_GESTAO', 'CO_NIVEL_HIERARQUIA', 'DS_NIVEL_HIERARQUIA', 'CO_ESFERA_ADMINISTRATIVA', 'DS_ESFERA_ADMINISTRATIVA', 'CO_ATIVIDADE', 'TP_UNIDADE', 'CO_CEP', 'NO_LOGRADOURO', 'NU_ENDERECO', 'NO_BAIRRO', 'NU_TELEFONE', 'NU_LATITUDE', 'NU_LONGITUDE', 'CO_TURNO_ATENDIMENTO', 'DS_TURNO_ATENDIMENTO', 'NU_CNPJ', 'NO_EMAIL', 'CO_NATUREZA_JUR', 'ST_CENTRO_CIRURGICO', 'ST_CENTRO_OBSTETRICO', 'ST_CENTRO_NEONATAL', 'ST_ATEND_HOSPITALAR', 'ST_SERVICO_APOIO', 'ST_ATEND_AMBULATORIAL', 'CO_MOTIVO_DESAB', 'CO_AMBULATORIAL_SUS']


In [41]:
merged_df.to_csv("../arquivos/mortes_mulheres_idade_fertil_com_cnes.csv", sep=';', index=False, encoding='utf-8')

In [42]:
# Verificação de quantos registros do mif não encontraram correspondência no CNES
mask_sem_cnes = ~mif['codigo_existe_no_cnes_num']
sem_correspondencia = mif[mask_sem_cnes]

print("=" * 70)
print("REGISTROS DO MIF SEM CORRESPONDÊNCIA NO CNES")
print("=" * 70)
print(f"Total de registros em mif:            {len(mif):>10,}")
print(f"Com correspondência no CNES:          {mif['codigo_existe_no_cnes_num'].sum():>10,} "
      f"({mif['codigo_existe_no_cnes_num'].mean()*100:.2f}%)")
print(f"SEM correspondência no CNES:          {mask_sem_cnes.sum():>10,} "
      f"({mask_sem_cnes.mean()*100:.2f}%)")

# Quantos não têm CODESTAB preenchido (explica parte da falta de correspondência)
sem_cnes_por_codestab_vazio = sem_correspondencia['CODESTAB'].isna().sum()
print(f"\n  Dos {len(sem_correspondencia):,} sem correspondência:")
print(f"    - Sem CODESTAB (vazio/NaN):        {sem_cnes_por_codestab_vazio:>10,}")
print(f"    - Com CODESTAB mas não no CNES:    {len(sem_correspondencia) - sem_cnes_por_codestab_vazio:>10,}")

REGISTROS DO MIF SEM CORRESPONDÊNCIA NO CNES
Total de registros em mif:               866,449
Com correspondência no CNES:             656,834 (75.81%)
SEM correspondência no CNES:             209,615 (24.19%)

  Dos 209,615 sem correspondência:
    - Sem CODESTAB (vazio/NaN):           204,754
    - Com CODESTAB mas não no CNES:         4,861


In [43]:
# Verificação de LOCOCOR dos registros que não encontraram correspondência no CNES
LOCOCOR_DESC = {
    '1': 'Hospital',
    '2': 'Outros est. saúde',
    '3': 'Domicílio',
    '4': 'Via pública',
    '5': 'Outros',
    '6': 'Aldeia indígena',
    '9': 'Ignorado',
}

mif['lococor_desc'] = mif['LOCOCOR'].map(LOCOCOR_DESC).fillna('(sem LOCOCOR)')
sem_correspondencia['lococor_desc'] = sem_correspondencia['LOCOCOR'].map(LOCOCOR_DESC).fillna('(sem LOCOCOR)')

print("--- Distribuição de LOCOCOR dos registros SEM correspondência no CNES ---")
print(sem_correspondencia['lococor_desc'].value_counts(dropna=False).to_string())

print("\n--- Cruzamento: correspondência no CNES × LOCOCOR ---")
print(pd.crosstab(mif['codigo_existe_no_cnes_num'].map({True: 'Com CNES', False: 'Sem CNES'}),
                  mif['lococor_desc'],
                  margins=True).to_string())

print("\n--- Percentual de LOCOCOR dentro de cada grupo ---")
tab = pd.crosstab(mif['codigo_existe_no_cnes_num'].map({True: 'Com CNES', False: 'Sem CNES'}),
                  mif['lococor_desc'], normalize='index')
print((tab * 100).round(1).to_string())

--- Distribuição de LOCOCOR dos registros SEM correspondência no CNES ---
lococor_desc
Domicílio            125179
Via pública           47846
Outros                30918
Hospital               4258
Ignorado                679
Outros est. saúde       620
Aldeia indígena         115

--- Cruzamento: correspondência no CNES × LOCOCOR ---
lococor_desc               Aldeia indígena  Domicílio  Hospital  Ignorado  Outros  Outros est. saúde  Via pública     All
codigo_existe_no_cnes_num                                                                                                
Com CNES                                 0          0    602320         0       0              54514            0  656834
Sem CNES                               115     125179      4258       679   30918                620        47846  209615
All                                    115     125179    606578       679   30918              55134        47846  866449

--- Percentual de LOCOCOR dentro de cada grupo ---
